# E1 · W9 · Kaggle run 20 — chạy ổn định 60 phút với cấu hình vận hành (GPU T4)
**Mục đích (W9-02 #130; `docs/w9/W9_PLAN.md` mục C):** cấu hình được chọn theo luật C (từ run 18, Claude áp luật ngoài Kaggle) chạy `e1pipe` liên tục 60 phút, lặp lại 38 video val; ghi RSS và bộ nhớ GPU mỗi 60 giây cùng mọi lỗi. Đạt khi không dừng do lỗi và RSS phút 60 ≤ 1,10 × RSS phút 10.

**Trước khi chạy:** điền `CONFIG` đúng cấu hình Claude báo sau run 18.


In [ ]:
COMMIT = ""                      # <-- SHA day du 40 ky tu (cung commit voi dataset e1-w9-tools)
CONFIG = ""                      # <-- "C0", "C1" hoac "C2" theo ket qua luat C (Claude bao sau run 18)
WEIGHTS_SHA = "a3de1e0195e4380aac497612296ed57f67e41ceb5ae3314e82a5ad66b125ef8a"
ULTRALYTICS_VERSION = "8.4.183"
RUNNER = "manual"
assert CONFIG in ("C0", "C1", "C2")


In [ ]:
import glob, hashlib, json, os, subprocess, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu (xem huong dan LOCAL)"
if not glob.glob("/kaggle/input/**/TOOLS_MANIFEST.json", recursive=True):
    zipfile.ZipFile(find("e1-w9-tools.zip")).extractall("/kaggle/working/tools")
TOOLS = str(Path((glob.glob("/kaggle/input/**/TOOLS_MANIFEST.json", recursive=True) or glob.glob("/kaggle/working/tools/**/TOOLS_MANIFEST.json", recursive=True))[0]).parent)
tm = json.load(open(f"{TOOLS}/TOOLS_MANIFEST.json"))
assert tm["commit"] == COMMIT, f"dataset tools dung commit {tm['commit'][:12]}, khac COMMIT — tao lai tools bang make_tools_bundle.py"
assert not tm["dirty_worktree"], "tools dung tu thu muc co thay doi chua commit"
bad = [f["path"] for f in tm["files"] if sha(f"{TOOLS}/{f['path']}") != f["sha256"]]
assert not bad, f"tep tools lech SHA: {bad[:5]}"
W = [p for p in sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True)) if sha(p) == WEIGHTS_SHA]
assert W, "Khong thay best.pt co SHA-256 cua yolo11n_640_s20261030 — kiem tra input e1-w5-run13"
WEIGHTS = W[0]
print("TOOLS", TOOLS, "| commit", tm["commit"][:12], "|", len(tm["files"]), "tep | WEIGHTS", WEIGHTS)


In [ ]:
!pip -q install "lap>=0.5.12" {"ultralytics==" + ULTRALYTICS_VERSION}
import lap, torch, ultralytics
print("ultralytics", ultralytics.__version__, "| lap", lap.__version__, "| cuda", torch.cuda.is_available())
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU T4 x2"


In [ ]:
# Du lieu tho Svanstrom: neu dataset la .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        if "svanstrom" in z.lower() or "drone-detection" in z.lower():
            print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
BENCH = f"{TOOLS}/scripts/kaggle/w9_e1pipe_bench.py"
print("DATA_ROOT =", DATA_ROOT)


In [ ]:
rc = subprocess.run([sys.executable, BENCH, "--data-root", DATA_ROOT, "--tools", TOOLS, "--weights", WEIGHTS, "--weights-sha", WEIGHTS_SHA,
                     "--config", CONFIG, "--rep", "1", "--duration-min", "60", "--out", "/kaggle/working", "--commit", COMMIT, "--runner", RUNNER]).returncode
print("ma thoat", rc)


## Tự kiểm tra (phải in `PASS`)

In [ ]:
d = f"/kaggle/working/w9_{CONFIG}_rep1_stab"
st = json.load(open(f"{d}/stage_status.json")); print({k: v["status"] for k, v in st.items()})
s = json.load(open(f"{d}/summary.json")); print(json.dumps(s.get("stability"), indent=1)); print(s["memory_samples"][-3:])
sb = s.get("stability") or {}
print("PASS" if rc == 0 and sb.get("minutes", 0) >= 60 and (sb.get("rss_ratio") or 9) <= 1.10 else "FAIL/KHONG DAT - gui stage_status.json va summary.json cho Claude")
z = f"/kaggle/working/w9_{CONFIG}_rep1_stab_evidence.zip"; print(z, os.path.getsize(z) // 1024, "KB  sha256", sha(z))
